In [8]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import warnings
warnings.filterwarnings('ignore')

# Use quote_plus to handle special characters in password
username = "postgres"
password = quote_plus("surya@2006")  # handles @ symbol safely
host     = "localhost"
port     = "5432"
database = "predictive_maintenance"

connection_string = f"postgresql://{username}:{password}@{host}:{port}/{database}"

engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print(f"Connected!")
        print(f"{result.fetchone()[0]}")
except Exception as e:
    print(f"Failed: {e}")

Connected!
PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit


In [10]:
try:
    with engine.connect() as conn:
        conn.execute(text(
            "CREATE EXTENSION IF NOT EXISTS timescaledb CASCADE;"
        ))
        conn.commit()
        print("TimescaleDB enabled!")
except Exception as e:
    print(f"Error: {e}")

TimescaleDB enabled!


In [12]:
create_table_sql = """
DROP TABLE IF EXISTS sensor_readings CASCADE;

CREATE TABLE sensor_readings (
    time            TIMESTAMPTZ      NOT NULL,
    equipment_id    INTEGER          NOT NULL,
    run_id          INTEGER          NOT NULL,
    cycle           INTEGER          NOT NULL,
    setting_1       DOUBLE PRECISION,
    setting_2       DOUBLE PRECISION,
    setting_3       DOUBLE PRECISION,
    sensor_1        DOUBLE PRECISION,
    sensor_2        DOUBLE PRECISION,
    sensor_3        DOUBLE PRECISION,
    sensor_4        DOUBLE PRECISION,
    sensor_5        DOUBLE PRECISION,
    sensor_6        DOUBLE PRECISION,
    sensor_7        DOUBLE PRECISION,
    sensor_8        DOUBLE PRECISION,
    sensor_9        DOUBLE PRECISION,
    sensor_10       DOUBLE PRECISION,
    sensor_11       DOUBLE PRECISION,
    sensor_12       DOUBLE PRECISION,
    sensor_13       DOUBLE PRECISION,
    sensor_14       DOUBLE PRECISION,
    sensor_15       DOUBLE PRECISION,
    sensor_16       DOUBLE PRECISION,
    sensor_17       DOUBLE PRECISION,
    sensor_18       DOUBLE PRECISION,
    sensor_19       DOUBLE PRECISION,
    sensor_20       DOUBLE PRECISION,
    sensor_21       DOUBLE PRECISION,
    failure_label   INTEGER          DEFAULT 0,
    rul             DOUBLE PRECISION,
    anomaly_score   DOUBLE PRECISION
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_table_sql))
        conn.commit()
        print("sensor_readings table created!")
except Exception as e:
    print(f"Error: {e}")

sensor_readings table created!


In [15]:
try:
    with engine.connect() as conn:

        # Create hypertable
        conn.execute(text("""
            SELECT create_hypertable(
                'sensor_readings',
                'time',
                chunk_time_interval => INTERVAL '7 days',
                if_not_exists => TRUE
            );
        """))
        conn.commit()
        print("Hypertable created! (7 day chunks)")

        # Enable compression
        conn.execute(text("""
            ALTER TABLE sensor_readings
            SET (
                timescaledb.compress,
                timescaledb.compress_orderby = 'time DESC'
            );
        """))
        conn.commit()
        print("Compression enabled!")

        # Add compression policy
        conn.execute(text("""
            SELECT add_compression_policy(
                'sensor_readings',
                INTERVAL '1 month',
                if_not_exists => TRUE
            );
        """))
        conn.commit()
        print("Compression policy set!")

except Exception as e:
    print(f"Error: {e}")

Hypertable created! (7 day chunks)
Compression enabled!
Compression policy set!


In [17]:
try:
    with engine.connect() as conn:
        conn.execute(text("""
            DROP TABLE IF EXISTS anomalies CASCADE;

            CREATE TABLE anomalies (
                time            TIMESTAMPTZ     NOT NULL,
                equipment_id    INTEGER         NOT NULL,
                anomaly_type    VARCHAR(50),
                severity        VARCHAR(20),
                anomaly_score   DOUBLE PRECISION,
                sensor_flagged  VARCHAR(50),
                message         TEXT,
                resolved        BOOLEAN DEFAULT FALSE
            );
        """))
        conn.commit()

        conn.execute(text("""
            SELECT create_hypertable(
                'anomalies',
                'time',
                if_not_exists => TRUE
            );
        """))
        conn.commit()
        print("Anomalies table created!")

except Exception as e:
    print(f"Error: {e}")

Anomalies table created!


In [18]:
import pandas as pd
import datetime

# Load data
cols = ['engine_id', 'cycle'] + \
       [f'setting_{i}' for i in range(1, 4)] + \
       [f'sensor_{i}' for i in range(1, 22)]

train_df = pd.read_csv(
    '../data/train_FD001.txt',
    sep='\s+',
    header=None,
    names=cols,
    engine='python'
)
train_df.dropna(axis=1, how='all', inplace=True)

# Add RUL
max_cycle = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycle.columns = ['engine_id', 'max_cycle']
train_df = train_df.merge(max_cycle, on='engine_id')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df.drop(columns=['max_cycle'], inplace=True)

# Add labels
train_df['failure_label'] = (train_df['RUL'] < 30).astype(int)
train_df['anomaly_score']  = 0.0

# Add timestamps
base_time = datetime.datetime(2024, 1, 1)
train_df['time'] = train_df.apply(
    lambda row: base_time + datetime.timedelta(
        hours=int((row['engine_id']-1)*500 + row['cycle'])
    ), axis=1
)

# Rename columns
train_df = train_df.rename(columns={'engine_id': 'equipment_id'})
train_df['run_id'] = train_df['equipment_id']

print(f"Data loaded: {train_df.shape}")

# Insert to database
try:
    train_df.to_sql(
        'sensor_readings',
        engine,
        if_exists='append',
        index=False,
        method='multi',
        chunksize=1000
    )
    print(f"{len(train_df)} rows inserted!")
except Exception as e:
    print(f"Insert error: {e}")


Data loaded: (20631, 31)
Insert error: (psycopg2.errors.UndefinedColumn) column "RUL" of relation "sensor_readings" does not exist
LINE 1: ...r_17, sensor_18, sensor_19, sensor_20, sensor_21, "RUL", fai...
                                                             ^

[SQL: INSERT INTO sensor_readings (equipment_id, cycle, setting_1, setting_2, setting_3, sensor_1, sensor_2, sensor_3, sensor_4, sensor_5, sensor_6, sensor_7, sensor_8, sensor_9, sensor_10, sensor_11, sensor_12, sensor_13, sensor_14, sensor_15, sensor_16, sensor_17, sensor_18, sensor_19, sensor_20, sensor_21, "RUL", failure_label, anomaly_score, time, run_id) VALUES (%(equipment_id_m0)s, %(cycle_m0)s, %(setting_1_m0)s, %(setting_2_m0)s, %(setting_3_m0)s, %(sensor_1_m0)s, %(sensor_2_m0)s, %(sensor_3_m0)s, %(sensor_4_m0)s, %(sensor_5_m0)s, %(sensor_6_m0)s, %(sensor_7_m0)s, %(sensor_8_m0)s, %(sensor_9_m0)s, %(sensor_10_m0)s, %(sensor_11_m0)s, %(sensor_12_m0)s, %(sensor_13_m0)s, %(sensor_14_m0)s, %(sensor_15_m0)s, %(sensor_1

In [20]:
queries = {
    "Total Rows"      : "SELECT COUNT(*) FROM sensor_readings;",
    "Unique Equipment": "SELECT COUNT(DISTINCT equipment_id) FROM sensor_readings;",
    "Date Range"      : "SELECT MIN(time), MAX(time) FROM sensor_readings;",
    "Anomaly Count"   : "SELECT COUNT(*) FROM sensor_readings WHERE failure_label = 1;",
    "Avg RUL"         : "SELECT ROUND(AVG(rul)::numeric, 2) FROM sensor_readings;"
}

print("📊 DATABASE SUMMARY")
print("=" * 50)
with engine.connect() as conn:
    for label, query in queries.items():
        result = conn.execute(text(query)).fetchone()
        print(f"{label:20s} : {result[0]}")
print("=" * 50)


📊 DATABASE SUMMARY
Total Rows           : 0
Unique Equipment     : 0
Date Range           : None
Anomaly Count        : 0
Avg RUL              : None
